In [ ]:
# block1
import pandas as pd
import numpy as np
import sys
import os
import joblib
import requests
from io import StringIO


from statsmodels.tsa.arima.model import ARIMA
import warnings

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import BayesianRidge 



from IPython.display import display

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

import plotly.express as px
import plotly.graph_objects as go

sys.path.append('..') 

# Safety check: Ensure our output folders exist before we save anything!
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

print("✅ Stage 1 Complete: Environment synchronized and directories ready.")

✅ Stage 1 Complete: Environment synchronized and directories ready.


In [ ]:
# block2


print("📡 Accessing Real Global Climate Data...")


co2_df = pd.read_csv('../data/raw/owid-co2-data.csv') 
ghg_df = pd.read_csv('../data/raw/total-ghg-emissions.csv') 

# 🚀 RENAME COLUMNS IMMEDIATELY so our filters don't crash!
ghg_df = ghg_df.rename(columns={
    'Entity': 'country',
    'Code': 'iso_code',
    'Year': 'year',
    'Annual greenhouse gas emissions including land use': 'Total_GHG'
})

# ---------------------------------------------------------
# 🚀 THE "ONE EARTH" FIX 
# We drop nulls and OWID aggregates immediately so they don't 
# cross-multiply and poison our merge.
# ---------------------------------------------------------
co2_df = co2_df.dropna(subset=['iso_code'])
co2_df = co2_df[~co2_df['iso_code'].str.startswith('OWID_')]

ghg_df = ghg_df.dropna(subset=['iso_code'])
ghg_df = ghg_df[~ghg_df['iso_code'].str.startswith('OWID_')]


# 2. Fetch REAL TEMPERATURE DATA (Bypassing OWID's Bot Protection)
temp_url = "https://ourworldindata.org/grapher/annual-temperature-anomalies.csv?v=1&csvType=full&useColumnShortNames=false"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(temp_url, headers=headers)

if response.status_code == 200:
    temp_df = pd.read_csv(StringIO(response.text))
else:
    raise Exception(f"❌ Server blocked the request! Status Code: {response.status_code}")

temp_df = temp_df.rename(columns={
    'Entity': 'country', 
    'Code': 'iso_code', 
    'Year': 'year', 
    'Temperature anomaly': 'Temp_Anomaly'
})

# 4. Merge the Science Data Together
# 🌟 ADDED 'methane' AND 'nitrous_oxide' TO THE MERGE!
df = pd.merge(co2_df[['country', 'iso_code', 'year', 'population', 'co2', 'methane', 'nitrous_oxide']], 
              ghg_df[['iso_code', 'year', 'Total_GHG']], 
              on=['iso_code', 'year'], how='left')

df = pd.merge(df, temp_df[['iso_code', 'year', 'Temp_Anomaly']], 
              on=['iso_code', 'year'], how='inner')

# 5. Rename for our Web App standard
df = df.rename(columns={
    'country': 'Real_Country_Name',
    'year': 'Year',
    'population': 'Population',
    'co2': 'CO2_Emissions',
    'methane': 'Methane_Emissions',             
    'nitrous_oxide': 'Nitrous_Oxide_Emissions'  
})

# 6. Cleanup & Baselines
df['Total_GHG'] = df['Total_GHG'].fillna(0)
df['CO2_Emissions'] = df['CO2_Emissions'].fillna(0)
df['Population'] = df['Population'].fillna(0)
df['Methane_Emissions'] = df['Methane_Emissions'].fillna(0)             
df['Nitrous_Oxide_Emissions'] = df['Nitrous_Oxide_Emissions'].fillna(0) 

df['Average_Temperature'] = 14.0 + df['Temp_Anomaly'] 

print(f"✅ Stage 2 Complete: Clean 'One Earth' Data Loaded! {len(df)} records merged.")

print("🔎 Preview of processed dataset:")
display(df.head(20))


📡 Accessing Real Global Climate Data...
✅ Stage 2 Complete: Clean 'One Earth' Data Loaded! 15725 records merged.
🔎 Preview of processed dataset:


,Real_Country_Name,iso_code,Year,Population,CO2_Emissions,Methane_Emissions,Nitrous_Oxide_Emissions,Total_GHG,Temp_Anomaly,Average_Temperature
0,Afghanistan,AFG,1940,6468510.0,0.000,6.814,1.240,16603607.0,-1.202316,12.797684
1,Afghanistan,AFG,1941,6528377.0,0.000,6.937,1.289,16855204.0,0.794745,14.794745
2,Afghanistan,AFG,1942,6608815.0,0.000,7.061,1.367,17041976.0,0.355437,14.355437
3,Afghanistan,AFG,1943,6710138.0,0.000,7.186,1.467,17262272.0,-1.005242,12.994758
4,Afghanistan,AFG,1944,6813016.0,0.000,7.312,1.583,17617690.0,-0.386346,13.613654
5,Afghanistan,AFG,1945,6917471.0,0.000,7.401,1.708,17872610.0,-1.163426,12.836574
6,Afghanistan,AFG,1946,7023527.0,0.000,7.490,1.835,18223516.0,-0.091273,13.908727
7,Afghanistan,AFG,1947,7131209.0,0.000,7.542,1.958,18634626.0,-0.186726,13.813274
8,Afghanistan,AFG,1948,7240542.0,0.000,7.631,2.069,18648814.0,-0.725991,13.274009
9,Afghanistan,AFG,1949,7356890.0,0.015,7.729,2.162,18666660.0,-2.020418,11.979582


In [ ]:
#block3


print("⚙️ Engineering Physics Features...")

# 1. MOVING AVERAGES
df['Temp_Moving_Avg'] = df.groupby('Real_Country_Name')['Temp_Anomaly'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
)

# 2. TIME DIMENSIONS
df['Decade'] = (df['Year'] // 10) * 10

# ---------------------------------------------------------
# 🌟 THE ARRHENIUS UPGRADE: LOGARITHMIC CARBON 🌟
# ---------------------------------------------------------
df = df.sort_values(by=['Real_Country_Name', 'Year'])
df['Cumulative_CO2'] = df.groupby('Real_Country_Name')['CO2_Emissions'].cumsum()

# We apply the natural log to mimic the physics of the atmosphere!
df['Log_Cumulative_CO2'] = np.log1p(df['Cumulative_CO2']) 

# 3. EXPORT THE PURE MASTER FILE
df.to_csv('../data/processed/supreme_dataset.csv', index=False)

print("✅ Stage 3 Complete: Arrhenius Physics injected and saved!")

⚙️ Engineering Physics Features...
✅ Stage 3 Complete: Arrhenius Physics injected and saved!


In [ ]:
# block4


print("🧠 Initializing the Bayesian Physics Workshop...")

# 1. CREATE GLOBAL DATA
global_df = df.groupby('Year').agg({
    'CO2_Emissions': 'sum',
    'Methane_Emissions': 'sum',           
    'Nitrous_Oxide_Emissions': 'sum',     
    'Population': 'sum',
    'Temp_Anomaly': 'mean', 
    'Cumulative_CO2': 'sum'
}).reset_index()

# 🚀 APPLY ARRHENIUS LOGARITHM
global_df['Log_Cumulative_CO2'] = np.log1p(global_df['Cumulative_CO2'])

# 🌊 OCEAN INERTIA (10-YEAR LAG)
global_df['Target_Temp_Anomaly'] = global_df['Temp_Anomaly'].shift(-10)
train_df = global_df.dropna(subset=['Target_Temp_Anomaly'])

# 2. PREPARE THE TRAINING DATA
features = ['Year', 'CO2_Emissions', 'Log_Cumulative_CO2', 'Population', 'Methane_Emissions', 'Nitrous_Oxide_Emissions']
X = train_df[features]
y = train_df['Target_Temp_Anomaly']

# 3. SCALE THE BRAIN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. TRAIN THE BAYESIAN PHYSICS ENGINE
print("⚡ Training the BayesianRidge Engine...")
ai_model = BayesianRidge() 
ai_model.fit(X_scaled, y)

# --- RE-ADDING ERROR METRICS ---


# Ask the AI to predict the past to see how well it learned
historical_predictions = ai_model.predict(X_scaled)

# Calculate the error
mae = mean_absolute_error(y, historical_predictions)
rmse = np.sqrt(mean_squared_error(y, historical_predictions))

print(f"📉 Mean Absolute Error (MAE): ±{mae:.3f}°C")
print(f"📉 Root Mean Squared Error (RMSE): ±{rmse:.3f}°C")
# -------------------------------

# 5. SAVE THE RECALIBRATED MODELS
joblib.dump(ai_model, '../models/supreme_nn_model.pkl') 
joblib.dump(scaler, '../models/supreme_scaler.pkl')
print("✅ Global AI trained and saved.")

# Note: We deleted Prophet! The Bayesian model handles it all now.

# 6. FEATURE IMPORTANCE
rf_explainer = RandomForestRegressor(random_state=42)
rf_explainer.fit(X, y)
importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_explainer.feature_importances_
})
importance_df.to_csv("../data/processed/feature_importance.csv", index=False)
print("🎉 PIPELINE COMPLETE! We are officially using Bayesian Probabilities.")

🧠 Initializing the Bayesian Physics Workshop...
⚡ Training the BayesianRidge Engine...
✅ Global AI trained and saved.
🎉 PIPELINE COMPLETE! We are officially using Bayesian Probabilities.


In [ ]:
#block5

warnings.filterwarnings('ignore')

print("📈 Generating ARIMA Statistical Baseline...")

# 1. Prepare the historical time-series data (Global average per year)
arima_df = df.groupby('Year')['Temp_Anomaly'].mean().reset_index()
arima_df = arima_df.sort_values('Year')
series = arima_df.set_index('Year')['Temp_Anomaly']

# 2. Fit the ARIMA(1, 1, 0) Model
print("⚡ Fitting ARIMA(1, 1, 0) Model...")
# 🚀 FIX: Add trend='t' to force the model to respect the upward drift!
model = ARIMA(series, order=(1, 1, 0), trend='t')
model_fit = model.fit()

# 3. Forecast into the future (e.g., next 60 years)
last_year = int(arima_df['Year'].max())
forecast_years = 60 
forecast = model_fit.get_forecast(steps=forecast_years)
forecast_mean = forecast.predicted_mean
conf_int = forecast.conf_int(alpha=0.05) # 95% CI

future_years = list(range(last_year + 1, last_year + forecast_years + 1))

# 4. Plot the result using Plotly (matching our app's UI)
fig_arima = go.Figure()

# Historical Data
fig_arima.add_trace(go.Scatter(
    x=arima_df['Year'], y=arima_df['Temp_Anomaly'],
    mode='lines', name='Historical Reality', line=dict(color='royalblue', width=2)
))

# Uncertainty Cone (Upper Bound - Invisible)
fig_arima.add_trace(go.Scatter(
    x=future_years, y=conf_int.iloc[:, 1],
    mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'
))

# Uncertainty Cone (Lower Bound - Filled)
fig_arima.add_trace(go.Scatter(
    x=future_years, y=conf_int.iloc[:, 0],
    mode='lines', fill='tonexty', fillcolor='rgba(255, 165, 0, 0.2)',
    line=dict(width=0), name='95% Confidence Interval'
))

# ARIMA Forecast
fig_arima.add_trace(go.Scatter(
    x=future_years, y=forecast_mean,
    mode='lines', name='ARIMA(1,1,0) Baseline', line=dict(color='darkorange', width=4, dash='dash')
))

# Danger Limit
fig_arima.add_hline(
    y=1.5, line_dash="dash", line_color="red", 
    annotation_text="1.5°C Danger Limit", annotation_position="bottom right"
)

fig_arima.update_layout(
    title="Baseline Statistical Forecast: ARIMA(1, 1, 0)",
    xaxis_title="Year",
    yaxis_title="Global Temp Anomaly (°C)",
    hovermode="x unified"
)

fig_arima.show()
print("✅ Block 5 Complete: ARIMA Diagram generated.")

print("🔍 Generating Feature Correlation Heatmap...")

# Select our core scientific features
corr_features = [
    'Temp_Anomaly', 'CO2_Emissions', 'Methane_Emissions', 
    'Nitrous_Oxide_Emissions', 'Population', 'Log_Cumulative_CO2'
]

# Calculate the Pearson correlation matrix
corr_matrix = df[corr_features].corr()

# Plot the heatmap using Plotly
fig_corr = px.imshow(
    corr_matrix, 
    text_auto='.2f', # Show the exact correlation numbers on the squares
    aspect="auto", 
    color_continuous_scale='RdBu_r', # Red for positive correlation, Blue for negative
    title="Global Climate Feature Correlation Matrix (Collinearity Check)"
)

fig_corr.show()
print("✅ Correlation Heatmap generated.")


# ---------------------------------------------------------
# 1. LINE PLOT — GLOBAL TEMPERATURE TREND
# ---------------------------------------------------------

line_df = df.groupby('Year')['Temp_Anomaly'].mean().reset_index()

fig_line = px.line(
line_df,
x='Year',
y='Temp_Anomaly',
title='Global Temperature Anomaly Trend Over Time',
markers=True
)

fig_line.update_layout(
xaxis_title='Year',
yaxis_title='Temperature Anomaly (°C)',
hovermode='x unified'
)

fig_line.show()

print("✅ Line Plot generated.")


# =========================================================
# 2. TIME SERIES DECOMPOSITION PLOT
# =========================================================

from statsmodels.tsa.seasonal import seasonal_decompose

print("📉 Generating Time Series Decomposition Plot...")

# Prepare yearly global temperature anomaly series

decomp_df = df.groupby('Year')['Temp_Anomaly'].mean().reset_index()
decomp_df = decomp_df.sort_values('Year')

series = decomp_df.set_index('Year')['Temp_Anomaly']

# Perform decomposition

decomposition = seasonal_decompose(
series,
model='additive',
period=10 # approximate long-term climate cycle
)

# Create Plotly figure

fig_decomp = go.Figure()

# Trend

fig_decomp.add_trace(go.Scatter(
x=series.index,
y=decomposition.trend,
mode='lines',
name='Trend'
))

# Seasonal

fig_decomp.add_trace(go.Scatter(
x=series.index,
y=decomposition.seasonal,
mode='lines',
name='Seasonal'
))

# Residual

fig_decomp.add_trace(go.Scatter(
x=series.index,
y=decomposition.resid,
mode='lines',
name='Residual'
))

fig_decomp.update_layout(
title='Climate Time Series Decomposition',
xaxis_title='Year',
yaxis_title='Value',
hovermode='x unified'
)

fig_decomp.show()

print("✅ Decomposition Plot generated.")

📈 Generating ARIMA Statistical Baseline...
⚡ Fitting ARIMA(1, 1, 0) Model...


✅ Block 5 Complete: ARIMA Diagram generated.
🔍 Generating Feature Correlation Heatmap...


✅ Correlation Heatmap generated.


✅ Line Plot generated.
📉 Generating Time Series Decomposition Plot...


✅ Decomposition Plot generated.
